In [1]:
!pip install jax optax rockpool

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.1/551.1 kB 337.0 kB/s  0:00:01 kB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached scipy-1.17.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 223.8 kB/s  0:00:13219.7 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 MB 1.3 MB/s  0:01:58 eta 0:00:010:00:020m
Using cached absl_py-2.4.0-py3-none-any.whl (135 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 1.4 MB/s  0:00:03m 1.4 MB/s eta 0:00:010m
Using cached scipy-1.17.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.2 MB)
Using cached opt_einsum-3.4.0-py3-none-any.whl (71 kB)
  Created wheel for rockpool: filenam

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import optax

from rockpool.nn.modules import Linear, LIF
from rockpool.nn.combinators import Sequential
from rockpool.training import make_jax

: 

In [ ]:
input_size = 8
hidden_size = 16
output_size = 2

model = Sequential(
    Linear((input_size, hidden_size)),
    LIF(hidden_size),
    Linear((hidden_size, output_size))
)

In [ ]:
jax_model, params = make_jax(model)

In [ ]:
def apply_resistor_mismatch(w, k=1.0, sigma=0.05):
    """
    w -> weights
    k -> scaling (R = k / w)
    sigma -> % mismatch (np. 5%)
    """

    # unikamy dzielenia przez zero
    w_safe = jnp.where(jnp.abs(w) < 1e-3, 1e-3, w)

    # mapowanie do rezystancji
    R = k / jnp.abs(w_safe)

    # losowy mismatch (proces technologiczny)
    noise = jax.random.normal(jax.random.PRNGKey(0), shape=R.shape) * sigma
    R_noisy = R * (1.0 + noise)

    # powrót do wag
    w_noisy = k / R_noisy

    # znak (dla wag ujemnych — differential)
    w_noisy = w_noisy * jnp.sign(w_safe)

    return w_noisy

In [ ]:
def apply_hardware_constraints(params):
    new_params = {}

    for layer_name, layer_params in params.items():
        new_layer = {}

        for p_name, p_value in layer_params.items():

            if "weight" in p_name:
                w_noisy = apply_resistor_mismatch(p_value)
                new_layer[p_name] = w_noisy
            else:
                new_layer[p_name] = p_value

        new_params[layer_name] = new_layer

    return new_params

In [ ]:
def loss_fn(params, x, y, key):
    # 🔥 hardware-aware weights
    params_noisy = apply_hardware_constraints(params)

    y_pred, _ = jax_model(params_noisy, x)

    loss = jnp.mean((y_pred - y) ** 2)
    return loss

In [ ]:
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

In [ ]:
@jax.jit
def train_step(params, opt_state, x, y, key):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y, key)

    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    return params, opt_state, loss

In [ ]:
# Dane do zmiany

T = 50

x = jnp.array(np.random.randn(T, input_size))
y = jnp.array(np.random.randn(T, output_size))

In [ ]:
key = jax.random.PRNGKey(42)

for epoch in range(100):

    key, subkey = jax.random.split(key)

    params, opt_state, loss = train_step(params, opt_state, x, y, subkey)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, loss: {loss}")